# OGC API Records search plugin

In this tutorial we will show you how to use EODAG to search data from providers exposing data through OGC API Records using [OARSearch](../../plugins_reference/generated/eodag.plugins.search.oar.OARSearch.rst) plugin.

In [1]:
from eodag import EODataAccessGateway

dag = EODataAccessGateway()

## Add a new provider

Add [geomet](https://api.weather.gc.ca) as new provider using [add_provider()](../../api_reference/core.rst#eodag.api.core.EODataAccessGateway.add_provider). 

GeoMet-OGC-API provides public access to the Meteorological Service of Canada (MSC) and Environment and Climate Change Canada (ECCC) data. 

Only search plugin type and `api_endpoint` are required, all remaining settings are kept as defaults.

In [2]:
dag.add_provider(
    "geomet", 
    search={
        "type": "OARSearch", 
        "api_endpoint": "https://api.weather.gc.ca",
    },
)

## Discover collections and queryables

List available collections for this provider:

In [3]:
dag.list_collections(provider="geomet")

id:,"'ahccd-annual',"
title:,"'AHCCD - Annual',"
description:,"'Adjusted and Homogenized Canadian Climate Data (AHCCD) are climate station datasets that incorporate adjustments (derived from statistical procedures) to the original historical station data to account for discontinuities from non-climatic factors, such as instrument changes or station relocation. Data are provided for temperature, precipitation, pressure and wind speed. Station trend data are provided when available. Trends are calculated using the Theil-Sen method using the station's full period of available data. The availability of trends will vary by station; if more than 5 consecutive years are missing data or more than 10% of the data within the time series is missing, a trend was not calculated.',"
extent:,"{ 'spatial': { 'bbox': [[-142 , 42 , -52 , 84 ] ] } , 'temporal': { 'interval': [[1840-01-01 00:00:00+00:00 , 2020-12-31 23:59:59.999999+00:00 ] ] } },"
keywords:,"['ahccd' , 'ahccd-annual' , 'annual' ],"
license:,"'other',"
id:,"'ahccd-monthly',"
title:,"'AHCCD - Monthly',"
description:,"'Adjusted and Homogenized Canadian Climate Data (AHCCD) are climate station datasets that incorporate adjustments (derived from statistical procedures) to the original historical station data to account for discontinuities from non-climatic factors, such as instrument changes or station relocation. Data are provided for temperature, precipitation, pressure and wind speed. Station trend data are provided when available. Trends are calculated using the Theil-Sen method using the station's full period of available data. The availability of trends will vary by station; if more than 5 consecutive years are missing data or more than 10% of the data within the time series is missing, a trend was not calculated.',"
extent:,"{ 'spatial': { 'bbox': [[-142 , 42 , -52 , 84 ] ] } , 'temporal': { 'interval': [[1840-01-01 00:00:00+00:00 , 2020-12-31 23:59:59.999999+00:00 ] ] } },"
keywords:,"['ahccd' , 'ahccd-monthly' , 'monthly' ],"


List queryable parameters for `climate-daily` collection:

In [4]:
dag.list_queryables(provider="geomet", collection="climate-daily")

QueryablesDict (36) - additional_properties=True
"str,"
"FieldInfo(annotation=NoneType, required=False, default=None, title='CLIMATE_IDENTIFIER')"
"int,"
"FieldInfo(annotation=NoneType, required=False, default=None, title='COOLING_DEGREE_DAYS')"
"str,"
"FieldInfo(annotation=NoneType, required=False, default=None, title='COOLING_DEGREE_DAYS_FLAG')"
"float,"
"FieldInfo(annotation=NoneType, required=False, default=None, title='DIRECTION_MAX_GUST')"
"str,"
"FieldInfo(annotation=NoneType, required=False, default=None, title='DIRECTION_MAX_GUST_FLAG')"


## Search data

Now search for `climate-daily` daµta on `2025-07-17` around southern Quebec. We'll use both queryable province code and a bounding box passed as geometry to select our Area Of Interest.

In [5]:
results = dag.search(
    provider="geomet",
    collection="climate-daily",
    start="2025-07-17", end="2025-07-17",
    geom=[-77, 45, -67, 50],
    PROVINCE_CODE="QC",
    limit=10, count=True,
)
results

SearchResult([EOProduct(id=7055121.2025.7.17, provider=geomet),
              EOProduct(id=7014160.2025.7.17, provider=geomet),
              EOProduct(id=7024280.2025.7.17, provider=geomet),
              EOProduct(id=7034482.2025.7.17, provider=geomet),
              EOProduct(id=7060400.2025.7.17, provider=geomet),
              EOProduct(id=7061541.2025.7.17, provider=geomet),
              EOProduct(id=7063370.2025.7.17, provider=geomet),
              EOProduct(id=7018563.2025.7.17, provider=geomet),
              EOProduct(id=7056616.2025.7.17, provider=geomet),
              EOProduct(id=7043BP9.2025.7.17, provider=geomet)])

Consume all pages to get all results:

In [6]:
from collections import deque

deque(results.next_page(update=True))

print(f"Got {len(results)} results")

Got 76 results


## Plot results on a map

Now plot on a map `TOTAL_PRECIPITATION` (mm) data from results properties.

In [7]:
import folium

fmap = folium.Map([47, -70], zoom_start=6)

# Create a layer that represents the search area in red
folium.Rectangle(
    bounds=[[45, -77], [50, -67]],
    color="red",
    tooltip="Search extent"
).add_to(fmap)

folium.GeoJson(
    data=results,
    marker=folium.Circle(radius=4, fill_color="orange", fill_opacity=0.4, color="black", weight=1),
    tooltip=folium.GeoJsonTooltip(fields=["geomet:STATION_NAME", "geomet:LOCAL_DATE", "geomet:TOTAL_PRECIPITATION"]),
    style_function=lambda x: {
        "fillColor": "blue",
        "radius": (x['properties']['geomet:TOTAL_PRECIPITATION'] or 0)*1000,
    },
    highlight_function=lambda x: {"fillOpacity": 0.8},
    zoom_on_click=True,
).add_to(fmap)

fmap